# 🧠 Production‑Grade ML System (Real‑World, End‑to‑End)

This notebook is a **cell‑by‑cell decomposition** of a **single production ML system**.

Each section contains:
- **What the step is**
- **Why it exists in real ML systems**
- The **exact code** (unchanged)

This is **not tutorial ML** — this is **deployable ML engineering**.


## Step 1️⃣ – Imports

**Why this step exists**

In production ML:
- Dependencies must be **explicit**
- No hidden notebook state
- Easy to freeze environments (`requirements.txt`)

This mirrors real deployment environments (Docker, CI/CD).


In [1]:

import os
import json
import joblib
import pandas as pd
import numpy as np
from datetime import datetime

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score


## Step 2️⃣ – Configuration & Data Contract

**Why this step exists**

A **data contract**:
- Prevents silent data corruption
- Defines allowed ranges and types
- Acts like an API contract for data

Without this, models fail *quietly*.


In [2]:

TARGET_COL = "target"

DATA_CONTRACT = {
    "age": {"type": "numeric", "min": 0, "max": 120},
    "salary": {"type": "numeric", "min": 0},
    "city": {"type": "categorical"},
}

MODEL_PATH = "models/full_pipeline.joblib"
LOG_PATH = "logs/predictions_log.csv"


## Step 3️⃣ – Load Data

**Why this step exists**

Separates **I/O** from **business logic**.
Allows future changes:
- CSV → Database
- Local → Cloud storage


In [3]:

def load_data(path):
    return pd.read_csv(path)


## Step 4️⃣ – Enforce Data Contract

**Why this step exists**

Handles:
- Strings in numeric columns
- Impossible values (age = 150)
- Missing mandatory columns

This is your **first line of defense**.


In [4]:

def enforce_contract(df, contract):
    df = df.copy()

    for col, rules in contract.items():
        if col not in df.columns:
            raise ValueError(f"Missing required column: {col}")

        if rules["type"] == "numeric":
            df[col] = pd.to_numeric(df[col], errors="coerce")

            if "min" in rules:
                df.loc[df[col] < rules["min"], col] = np.nan
            if "max" in rules:
                df.loc[df[col] > rules["max"], col] = np.nan

        elif rules["type"] == "categorical":
            df[col] = df[col].astype(str)

    return df


## Step 5️⃣ – Detect Column Types (Never Trust dtypes)

**Why this step exists**

Real datasets:
- Mix strings and numbers
- Lie about dtypes
- Change over time

This detects **intent**, not pandas dtype.


In [5]:

def detect_column_types(df):
    numeric_cols, categorical_cols = [], []

    for col in df.columns:
        if col == TARGET_COL:
            continue

        coerced = pd.to_numeric(df[col], errors="coerce")
        numeric_ratio = coerced.notnull().mean()

        if numeric_ratio > 0.7:
            numeric_cols.append(col)
        else:
            categorical_cols.append(col)

    return numeric_cols, categorical_cols


## Step 6️⃣ – Leakage Detection

**Why this step exists**

High accuracy can be **fake** due to:
- Post‑event features
- ID leakage
- Future information

Senior engineers *never* trust accuracy blindly.


In [6]:
def leakage_check(X, y, threshold=0.95):
    """
    Detects potential target leakage by checking
    correlation between numeric features and the target.
    """
    if not np.issubdtype(y.dtype, np.number):
        return

    numeric_X = X.select_dtypes(include=[np.number])

    if numeric_X.empty:
        return

    correlations = {}

    for col in numeric_X.columns:
        corr = np.corrcoef(numeric_X[col], y)[0, 1]
        if not np.isnan(corr):
            correlations[col] = abs(corr)

    suspicious = {
        col: corr
        for col, corr in correlations.items()
        if corr > threshold
    }

    if suspicious:
        print("⚠️ Potential target leakage detected:")
        for col, corr in suspicious.items():
            print(f"  {col}: correlation={corr:.3f}")


## Step 7️⃣ – Baseline Sanity Check

**Why this step exists**

If ML cannot beat:
> "Always predict the majority class"

Then ML **should not be used**.


In [7]:

def majority_baseline(y):
    return y.value_counts(normalize=True).iloc[0]


## Step 8️⃣ – Build Preprocessing Pipeline

**Why this step exists**

Ensures:
- Same preprocessing in training & inference
- No leakage
- No manual hacks

This is **industry standard**.


In [8]:

def build_preprocessor(numeric_cols, categorical_cols):

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        ))
    ])

    return ColumnTransformer([
        ("num", numeric_pipeline, numeric_cols),
        ("cat", categorical_pipeline, categorical_cols)
    ])


## Step 9️⃣ – Train Full ML System

**Why this step exists**

This is the **core orchestration**:
- Validation
- Baseline
- Leakage checks
- Training
- Evaluation
- Persistence


In [9]:

def train_system(data_path):

    os.makedirs("models", exist_ok=True)
    os.makedirs("logs", exist_ok=True)

    df = load_data(data_path)
    df = enforce_contract(df, DATA_CONTRACT)

    X = df.drop(columns=[TARGET_COL])
    y = df[TARGET_COL]

    baseline = majority_baseline(y)
    print(f"Baseline accuracy: {baseline:.4f}")

    numeric_cols, categorical_cols = detect_column_types(df)
    leakage_check(X, y)

    X_train, X_val, y_train, y_val = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    preprocessor = build_preprocessor(numeric_cols, categorical_cols)

    model = RandomForestClassifier(
        n_estimators=300,
        max_depth=12,
        random_state=42,
        n_jobs=-1
    )

    pipeline = Pipeline([
        ("preprocessing", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)

    preds = pipeline.predict(X_val)
    acc = accuracy_score(y_val, preds)
    print(f"Model accuracy: {acc:.4f}")

    joblib.dump(pipeline, MODEL_PATH)

    metadata = {
        "trained_at": datetime.utcnow().isoformat(),
        "accuracy": acc,
        "baseline": baseline,
        "numeric_cols": numeric_cols,
        "categorical_cols": categorical_cols
    }

    with open("models/metadata.json", "w") as f:
        json.dump(metadata, f, indent=2)

    return pipeline


## Step 🔟 – Inference Guardrails

**Why this step exists**

Prevents:
- Missing columns
- Silent prediction corruption


In [10]:

def validate_inference_input(df, pipeline):
    expected = pipeline.feature_names_in_
    missing = set(expected) - set(df.columns)

    if missing:
        raise ValueError(f"Inference missing columns: {missing}")


## Step 1️⃣1️⃣ – Prediction with Confidence & Abstention

**Why this step exists**

Real systems must:
- Know when they are unsure
- Fail safely instead of guessing


In [11]:
def predict(raw_df, threshold=0.6):

    pipeline = joblib.load(MODEL_PATH)

    raw_df = enforce_contract(raw_df, DATA_CONTRACT)
    validate_inference_input(raw_df, pipeline)

    # Model probabilities
    probs = pipeline.predict_proba(raw_df)
    max_probs = probs.max(axis=1)

    # Raw model predictions (numeric, untouched)
    model_preds = pipeline.classes_[probs.argmax(axis=1)]

    # Business-level decisions (string-safe)
    final_preds = []

    for pred, conf in zip(model_preds, max_probs):
        if conf < threshold:
            final_preds.append("ABSTAIN")
        else:
            final_preds.append(int(pred))

    log_predictions(raw_df, final_preds, max_probs)

    return final_preds, max_probs



## Step 1️⃣2️⃣ – Logging (Monitoring Hook)

**Why this step exists**

If predictions are not logged:
- You cannot debug
- You cannot detect drift
- You cannot trust the system


In [12]:

def log_predictions(df, preds, probs):
    log_df = df.copy()
    log_df["prediction"] = preds
    log_df["confidence"] = probs
    log_df["timestamp"] = datetime.utcnow().isoformat()

    header = not os.path.exists(LOG_PATH)
    log_df.to_csv(LOG_PATH, mode="a", header=header, index=False)


## Step 1️⃣3️⃣ – Entry Point

**Why this step exists**

Separates:
- Training workflow
- Inference usage


In [13]:

if __name__ == "__main__":

    train_system("data/raw/data.csv")

    sample = pd.DataFrame({
        "age": ["34", "unknown", 150],
        "salary": [50000, "60000", None],
        "city": ["Delhi", 123, "Mumbai"]
    })

    predictions, confidence = predict(sample)
    print("Predictions:", predictions)
    print("Confidence:", confidence)


Baseline accuracy: 0.5000
Model accuracy: 1.0000
Predictions: [1, 0, 0]
Confidence: [0.60666667 0.74333333 0.77      ]


/tmp/ipykernel_7397/3352389770.py:48: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "trained_at": datetime.utcnow().isoformat(),
/tmp/ipykernel_7397/4045137793.py:5: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  log_df["timestamp"] = datetime.utcnow().isoformat()
